# 03 — 失败案例分析 & Prompt 消融

本 Notebook 涵盖：
1. 自动筛选失败案例并按类型归类
2. 典型失败案例可视化
3. Prompt 模板消融实验（OVD & VG）

In [ ]:
import sys, random
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

from src.utils.io import load_json, load_yaml
from src.analysis.failure_miner import mine_failures, summarize_failures, select_examples
from src.analysis.visualize import visualize_vg_sample

random.seed(42)
print('准备就绪')

## 1. 失败案例统计

In [ ]:
PRED_FILE = '../results/refcoco/refcoco_val_predictions.json'

per_sample = load_json(PRED_FILE)
failures = mine_failures(per_sample)
summary = summarize_failures(failures)

n_total = len(per_sample)
n_fail  = len(failures)
acc = (n_total - n_fail) / n_total * 100

print(f'总样本：{n_total}，失败：{n_fail}，Acc@0.5 = {acc:.2f}%')
print('失败类型分布：', summary)

In [ ]:
# 失败类型饼图
labels = list(summary.keys())
sizes  = list(summary.values())

fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.set_title(f'RefCOCO val 失败案例类型分布\n（共 {n_fail} 个失败）')
plt.tight_layout()
plt.savefig('../results/refcoco/failure_type_pie.png', dpi=120)
plt.show()

## 2. 典型失败案例可视化

In [ ]:
def show_failure_grid(examples, title='失败案例', cols=5):
    rows = (len(examples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    if rows == 1:
        axes = [axes]
    for ax, sample in zip([a for row in axes for a in (row if hasattr(row,'__iter__') else [row])], examples):
        try:
            pil_img = Image.open(sample['img_path']).convert('RGB')
        except Exception:
            ax.set_visible(False)
            continue
        ax.imshow(pil_img)
        ax.set_title(
            f"IoU={sample.get('iou',0):.2f}  score={sample.get('pred_score',0):.2f}\n"
            f"[{', '.join(sample.get('failure_types',['?']))}]\n"
            f"{sample.get('expr','')[:45]}",
            fontsize=7,
        )
        ax.axis('off')
        if sample.get('pred_box'):
            x1,y1,x2,y2 = sample['pred_box']
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='red',fc='none'))
        if sample.get('gt_box'):
            x1,y1,x2,y2 = sample['gt_box']
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec='lime',fc='none',ls='--'))
    # 隐藏多余的子图
    for ax in [a for row in axes for a in (row if hasattr(row,'__iter__') else [row])][len(examples):]:
        ax.set_visible(False)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    return fig

In [ ]:
# 小目标失败
small_obj_fails = select_examples(failures, failure_type='small_obj', n=5)
if small_obj_fails:
    fig = show_failure_grid(small_obj_fails, '失败类型：小目标 (small_obj)')
    plt.savefig('../results/refcoco/failure_small_obj.png', dpi=100)
    plt.show()

In [ ]:
# 语言歧义失败
lang_fails = select_examples(failures, failure_type='lang_ambig', n=5)
if lang_fails:
    fig = show_failure_grid(lang_fails, '失败类型：语言歧义 (lang_ambig)')
    plt.savefig('../results/refcoco/failure_lang_ambig.png', dpi=100)
    plt.show()

In [ ]:
# 高置信度但失败（模型最自信的错误案例）
confident_fails = select_examples(failures, n=5, sort_by='score_desc')
if confident_fails:
    fig = show_failure_grid(confident_fails, '高置信度失败案例（score 最高但 IoU < 0.5）')
    plt.savefig('../results/refcoco/failure_confident.png', dpi=100)
    plt.show()

## 3. Prompt 消融实验

In [ ]:
from src.model_wrapper import GroundingDINOWrapper
from src.ovd.coco_prompt_builder import build_concat_prompt, COCO80_NAMES

cfg = load_yaml('../configs/coco_ovd.yaml')
model = GroundingDINOWrapper.from_config(cfg)
print('模型加载完成')

In [ ]:
# 3a. OVD Prompt 模板消融：在少量图像上对比三种模板
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from src.ovd.coco_prompt_builder import build_name_to_catid, match_phrase_to_catid
from src.utils.box_ops import xyxy_to_xywh
import json

ANN_FILE = '../data/coco/annotations/instances_val2017.json'
IMG_DIR  = Path('../data/coco/val2017')

coco_gt = COCO(ANN_FILE)
name2catid = build_name_to_catid(coco_gt)

# 使用少量图像做快速消融
ABLATION_N = 200
random.seed(42)
ablation_ids = random.sample(coco_gt.getImgIds(), ABLATION_N)

templates = ['{name}', 'a {name}', 'a photo of a {name}']
ablation_results = {}

for tmpl in templates:
    prompt = build_concat_prompt(COCO80_NAMES, template=tmpl)
    preds = []
    for img_id in ablation_ids:
        img_info = coco_gt.loadImgs(img_id)[0]
        img_path = IMG_DIR / img_info['file_name']
        try:
            pil_img = Image.open(img_path).convert('RGB')
        except Exception:
            continue
        result = model.predict(pil_img, prompt)
        if len(result.boxes_xyxy) == 0:
            continue
        boxes_xywh = xyxy_to_xywh(result.boxes_xyxy)
        for box, score, phrase in zip(boxes_xywh, result.scores, result.phrases):
            cat_id = match_phrase_to_catid(phrase, name2catid)
            if cat_id is None:
                continue
            preds.append({'image_id': img_id, 'category_id': cat_id,
                          'bbox': box.tolist(), 'score': float(score)})

    tmp_pred_file = f'/tmp/ablation_{tmpl[:5].replace(" ","_")}.json'
    with open(tmp_pred_file, 'w') as f:
        json.dump(preds, f)

    if preds:
        coco_dt = coco_gt.loadRes(tmp_pred_file)
        ev = COCOeval(coco_gt, coco_dt, 'bbox')
        ev.params.imgIds = ablation_ids
        ev.evaluate(); ev.accumulate(); ev.summarize()
        ablation_results[tmpl] = {'mAP': ev.stats[0], 'AP50': ev.stats[1]}
    else:
        ablation_results[tmpl] = {'mAP': 0.0, 'AP50': 0.0}

print('\n== OVD Prompt 消融结果 ==')
for tmpl, m in ablation_results.items():
    print(f'  [{tmpl}]  mAP={m["mAP"]*100:.1f}  AP50={m["AP50"]*100:.1f}')

In [ ]:
# 3b. VG Prompt 消融：完整指代句 vs 去掉空间关系词
import re

SPATIAL_WORDS = [
    r'\b(left|right|top|bottom|above|below|behind|front|near|far|'  
    r'center|middle|corner|inside|outside|between)\b',
    r'\b(next to|on top of|in front of|to the (left|right) of)\b',
]

def strip_spatial(expr: str) -> str:
    for pattern in SPATIAL_WORDS:
        expr = re.sub(pattern, '', expr, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', expr).strip()

# 选取包含空间词的失败 / 成功样本各 50 条做对比
has_spatial = [s for s in per_sample
               if any(re.search(p, s.get('expr',''), re.IGNORECASE) for p in SPATIAL_WORDS)]
sample_vg_ablation = random.sample(has_spatial, min(50, len(has_spatial)))

hits_full, hits_stripped = 0, 0
for sample in sample_vg_ablation:
    pil_img = Image.open(sample['img_path']).convert('RGB')
    gt_box  = np.array(sample['gt_box'])

    # 完整表达式
    r1 = model.predict(pil_img, sample['expr'])
    if len(r1.boxes_xyxy) > 0:
        from src.utils.box_ops import batch_iou
        iou1 = batch_iou(r1.boxes_xyxy, gt_box)[np.argmax(r1.scores)]
        hits_full += int(iou1 >= 0.5)

    # 去掉空间关系词
    stripped_expr = strip_spatial(sample['expr'])
    if stripped_expr:
        r2 = model.predict(pil_img, stripped_expr)
        if len(r2.boxes_xyxy) > 0:
            iou2 = batch_iou(r2.boxes_xyxy, gt_box)[np.argmax(r2.scores)]
            hits_stripped += int(iou2 >= 0.5)

n = len(sample_vg_ablation)
print(f'\n== VG Prompt 消融结果（n={n}）==')
print(f'  完整指代句      Acc@0.5 = {hits_full/n*100:.1f}%')
print(f'  去掉空间关系词  Acc@0.5 = {hits_stripped/n*100:.1f}%')
print('  结论：空间关系词对定位精度的影响...')

In [ ]:
# 汇总消融结果为表格
import pandas as pd

ovd_rows = [{'实验': f'OVD Prompt: {t}',
             'mAP': f"{ablation_results[t]['mAP']*100:.1f}",
             'AP50': f"{ablation_results[t]['AP50']*100:.1f}"}
            for t in templates]
vg_rows = [
    {'实验': 'VG 完整指代句', 'Acc@0.5': f'{hits_full/n*100:.1f}%'},
    {'实验': 'VG 去掉空间关系词', 'Acc@0.5': f'{hits_stripped/n*100:.1f}%'},
]

print('=== OVD Prompt 消融 ===')
display(pd.DataFrame(ovd_rows))
print('\n=== VG Prompt 消融 ===')
display(pd.DataFrame(vg_rows))